# Stage 3.5 — INLP / LEACE causal intervention with layer sweep

For each model in the probing subset, fit linear-concept-erasure projections (INLP + LEACE) **at multiple layer depths**, sanity-check, then re-run CrowS-Pairs and StereoSet with the projection hook attached at each depth.

**Why a layer sweep?** Linear probes for demographic attributes peak in early layers (mostly surface keyword detection from the labelled-by-keyword data). But the *causal effect* of ablating those directions on output bias peaks at different (typically middle) depths, where the model actually uses the demographic info. Reporting across depths lets us identify the **functional locus** of bias representation, distinct from the probe-accuracy locus — a key methodological point in the thesis.

Default sweep: depths 10%, 30%, 50%, 70%, 90% (5 layers per model).

**Prereq:** activations + probe results from `run_probing.py` for each probing-subset model (we read activations per layer from `results/activations/<model>/`).

**Workflow:** validate first on one model → inspect sanity reports across depths → then commit to the full sweep.

In [ ]:
GITHUB_REPO = 'https://github.com/korneli777/llm-bias-eval.git'
DRIVE_DIR = '/content/drive/MyDrive/llm-bias-eval'

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata
from huggingface_hub import login

REPO_DIR = '/content/llm-bias-eval'

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

results_link = os.path.join(REPO_DIR, 'results')
if os.path.lexists(results_link):
    (os.unlink if os.path.islink(results_link) else shutil.rmtree)(results_link)
os.symlink(f'{DRIVE_DIR}/results', results_link)

subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'llm-bias-eval'
login(token=os.environ['HF_TOKEN'])

## Step 1 — Validate on Llama-3.1-8B (one model, gender, both methods, all 5 depths)

`--validate-only` fits the projections + runs both sanity checks (probe nullification, LM perplexity blow-up) at every depth in the sweep but **does not** re-run benchmarks. Inspect the sanity reports under `results/intervention/<model>/<attr>__<method>__L<NN>__sanity.json` before committing to the full sweep.

Pass criteria per depth:
- `nullification.passed = True` (post-projection probe accuracy ≤ 0.55)
- `perplexity.passed = True` (post/pre ratio ≤ 1.5×)

If perplexity blows up at shallow depths (e.g. L00, L02), that's expected — projecting out a direction near the embedding can disrupt the model. Middle/late layers should pass cleanly.

In [ ]:
!python scripts/run_intervention.py \
    --models meta-llama/Llama-3.1-8B \
    --attributes gender \
    --methods inlp leace \
    --validate-only \
    --no-wandb

In [ ]:
# Inspect the sanity reports inline (one per layer depth).
import json
from pathlib import Path
sanity_files = sorted(Path('results/intervention/meta-llama__Llama-3.1-8B').glob('*__sanity.json'))
print(f"{'file':50s}  {'null_pass':>10s}  {'ppl_pre':>9s}  {'ppl_post':>9s}  {'ratio':>6s}  {'ppl_pass':>9s}")
print('-' * 110)
for fp in sanity_files:
    info = json.load(open(fp))
    n = info.get('nullification', {})
    p = info.get('perplexity', {})
    print(f"  {fp.name:48s}  {str(n.get('passed')):>10s}  "
          f"{p.get('perplexity_pre', 0):>9.2f}  {p.get('perplexity_post', 0):>9.2f}  "
          f"{p.get('ratio', 0):>6.2f}  {str(p.get('passed')):>9s}")

## Step 2 — Full sweep (only after validation passes)

16 probe-subset models × 2 attributes (gender, race) × 5 depths × 2 benchmarks (CrowS-Pairs, StereoSet) × 2 prompt_modes (raw, instruct) × 2 methods (INLP, LEACE) ≈ **2,560 cells** at full coverage.

Resumable per cell — `is_completed()` skips finished JSONs. Estimated ~30-40 hours total on H100; safe to interrupt and re-fire any time.

Tip: if you want to validate the layer-sweep idea quickly first, drop one of these flags to limit scope:
- `--methods inlp` (skip LEACE, halves the work)
- `--benchmarks crows_pairs` (skip StereoSet, halves the work)
- `--layer-depths 0.5` (just middle layer)

In [ ]:
!python scripts/run_intervention.py \
    --benchmarks crows_pairs stereoset \
    --attributes gender race \
    --methods inlp leace \
    --prompt-modes raw instruct